**Українська версія:**

Дослідницький Аналіз Даних (EDA)

У цьому ноутбуці проводиться дослідницький аналіз даних клієнтів телекомунікаційної компанії з метою розуміння основних закономірностей та характеристик набору даних перед побудовою моделі прогнозування відтоку.

Завдання:

1. Вивчення розподілів ключових ознак (демографічні дані, тривалість користування послугами, тарифи, тип контракту тощо)
2. Виявлення відсутніх значень та оцінка їхнього впливу на набір даних
3. Аналіз кореляцій між ознаками та цільовою змінною (Churn)
4. Виявлення потенційних викидів та проблем якості даних, які потребуватимуть уваги на етапі попередньої обробки

Висновки, отримані на цьому етапі, безпосередньо визначатимуть кроки попередньої обробки даних на наступному етапі проєкту.

**English version:**

Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on the telecom customer dataset to understand the underlying patterns and characteristics before building a churn prediction model.

Goals:

1. Explore the distribution of key features (demographics, tenure, charges, contract types, etc.)
2. Identify missing values and assess their impact on the dataset
3. Analyze correlations between features and the target variable (Churn)
4. Detect potential outliers and data quality issues that may require attention during preprocessing

The insights gathered here will directly inform the preprocessing steps in the next stage of the project.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#### Розділ функцій (Functions section) 

In [ ]:
    
def get_churn_distribution(data_structure,min_value, max_value,column_comparison, interval,procedural_slip) -> dict:
    """ 
    Українська версія: Функція обчислює кількість випадків відтоку для кожного кроку в заданій колонці набору даних 
    і повертає словник, що містить набір записів по кроках, кожен з яких містить інтервал колонки та відповідну кількість відтоку.

    English version: The function calculates the quantity of churns per step in a given dataset column, 
    and returns a dictionary containing a set of step-level entries, each with the column interval and 
    the corresponding churn quantity.
    """
    dict_values = {}

    previous_ind = 0
    attempt = 0
    i = min_value
    
    while (i <= max_value + procedural_slip):
    
        if attempt == 0:
            column_value = i
            churn_quantity = data_structure.loc[data_structure[column_comparison] <= i, 'churn'].sum()
            attempt = 1
        else:
            column_value = i
            churn_quantity = data_structure.loc[(data_structure[column_comparison] > previous_ind) & (data_structure[column_comparison] <= i), 'churn'].sum()
            
        list_values = {}
        final_result = {f'{column_comparison}': column_value, 'churn_quantity': churn_quantity}
        dict_values[f'{i}_step:'] = final_result
        previous_ind = i
        i += interval
        
    return dict_values

In [ ]:
def return_results_as_lists(min_value, max_value, column_comparison, dictionary,interval,procedural_slip):
    """ 
    Українська версія: Функція повертає два списки для спрощення подальших обчислень або графічного представлення 
    залежності між інтервалами колонки та кількістю відтоку.
    
    English version: The function returns two lists for simpler further calculations or graphical representation 
    of the relationship between column intervals and churn quantity.
    """ 
    i = min_value
    list_column_values = []
    list_churn_quantity = []
    
    while (i <= max_value + procedural_slip):
        list_column_values.append(dictionary[f'{i}_step:'][column_comparison])
        list_churn_quantity.append(dictionary[f'{i}_step:']['churn_quantity'])
        i += interval
        
    return list_column_values,list_churn_quantity

In [ ]:
def show_all_churn_piecharts(data_structure, request_column: str, usage_labels: list, churn_labels:list, all_title: str, churns_title: str, with_churns_title: str,without_churns_title: str) -> None:
    """
    Українська версія:
    Будує чотири графіки для заданої ознаки (`request_column`):

    1. Використання ознаки серед УСІХ клієнтів. Показує, який відсоток усіх клієнтів мав ненульове значення
       заданої ознаки, а який — не мав.

    2. Використання ознаки серед КЛІЄНТІВ, ЯКІ ПІШЛИ. Показує, який відсоток клієнтів, що пішли, мав ненульове значення 
       заданої ознаки, а який — не мав. Тобто графік показує структуру групи клієнтів, які пішли. 
       Він НЕ показує, що ця ознака була причиною відтоку.

    3. Рівень відтоку серед клієнтів, ЯКІ МАЛИ ознаку. Показує, який відсоток клієнтів із ненульовим значенням
       заданої ознаки пішов, а який залишився активним.

    4. Рівень відтоку серед клієнтів, ЯКІ НЕ МАЛИ ознаки. Показує, який відсоток клієнтів із нульовим значенням 
       заданої ознаки пішов, а який залишився активним.
    
    English version: 
    Builds four pie charts for the specified feature (`request_column`):

    1. Feature usage among ALL customers. Shows what percentage of all customers had a non-zero value
       in the specified feature and what percentage did not.

    2. Feature usage among CHURNED customers. Shows what percentage of customers who churned had a non-zero value 
       in the specified feature and what percentage did not. This describes the composition of the churned group.
       It does NOT show that the feature caused the churn.

    3. Churn rate among customers WITH the feature. Shows what percentage of customers who had a non-zero value
       in the specified feature churned and what percentage remained active.

    4. Churn rate among customers WITHOUT the feature. Shows what percentage of customers who had a zero value
   in the specified feature churned and what percentage remained active.
    """

    customers_with_feature = (data_structure[request_column] > 0).sum()
    customers_without_feature = (data_structure[request_column] == 0).sum()
    
    churned_with_feature = data_structure[data_structure[request_column] > 0]['churn'].sum()
    churned_without_feature = data_structure[data_structure[request_column] == 0]['churn'].sum()

    active_with_feature = customers_with_feature - churned_with_feature
    active_without_feature = customers_without_feature - churned_without_feature

    fig, axs = plt.subplots(2, 2, figsize=(14,9))
    axs = axs.flatten()

    data_all = [customers_with_feature, customers_without_feature]
    axs[0].set_title(all_title)
    axs[0].pie(data_all,labels=usage_labels,autopct="%.2f%%",colors=['Teal','Salmon'],shadow=False, labeldistance=1.1, startangle=0, radius=1)

    data_churns = [churned_with_feature, churned_without_feature]
    axs[1].set_title(churns_title)
    axs[1].pie(data_churns,labels=usage_labels,autopct="%.2f%%",colors=['green','orange'],shadow=False, labeldistance=1.1, startangle=0, radius=1)

    customers_with_the_feature = [churned_with_feature, active_with_feature]
    axs[2].set_title(with_churns_title)
    axs[2].pie(customers_with_the_feature,labels=churn_labels,autopct="%.2f%%",colors=['green','orange'],shadow=False, labeldistance=1.1, startangle=0, radius=1)

    customers_without_the_feature = [churned_without_feature, active_without_feature]
    axs[3].set_title(without_churns_title)
    axs[3].pie(customers_without_the_feature,labels=churn_labels,autopct="%.2f%%",colors=['Teal','Salmon'],shadow=False, labeldistance=1.1, startangle=0, radius=1)
    plt.subplots_adjust(hspace=0.4, wspace=0.3)
    # plt.tight_layout()
    plt.show()

In [ ]:
def show_graph_and_table_churns(churn_list: list,input_list: list, title:str,x_label:str,table_height: float = 1.4) -> None:
    total_churn = sum(churn_list)
    cell_text = [['%1.2f' % ((x / total_churn) * 100)] for x in churn_list]

    plt.figure(figsize=(9, 5))
    plt.plot(input_list,churn_list)
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel('Кількість відтоку')
        
    the_table = plt.table(cellText=cell_text,rowLabels=input_list,colLabels=['Відтік y %'], loc='left',bbox=[-0.45, 0, 0.2, table_height] )
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(10)
    plt.show()

In [ ]:
df = pd.read_csv('../data/raw/internet_service_churn.csv')

In [ ]:
df.head(10)

**Українська версія:**    
Помилку в назві стовпця, а саме «reamining_contract», необхідно виправити, надавши їй правильне значення — «remaining contract»

**English version:**   
The error in the column name, specifically “reamining_contract,” must be corrected by replacing it with the correct value —“remaining contract.”

In [ ]:
df = df.rename(columns={'reamining_contract': 'remaining_contract'})

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df[df['subscription_age'] < 0]

In [ ]:
df.loc[df['subscription_age'] < 0, 'subscription_age'] = 0

In [ ]:
duplications_amount = df.duplicated().sum()
if duplications_amount > 0:
    df = df.drop_duplicates()
    df
duplications_amount

In [ ]:
churn_percentage = round(df['churn'].mean() * 100,2)
print(f'рівень відтоку клієнтів становить {churn_percentage}% з набору даних')
# print(f'The customer churn rate is {churn_percentage} from dataset')

In [ ]:
df.head(15)

In [ ]:
df['is_tv_subscriber'].value_counts()

In [ ]:
df['is_movie_package_subscriber'].value_counts()

**Українська версія:**  
Оскільки в наборі даних представлені лише окремі значення щодо наявності підписок на телевізійні та кінопакети, варто з’ясувати кількість користувачів, які мають обидві підписки, оскільки це може вплинути на потенційний відтік клієнтів.

**English Version:**    
Since the dataset contains only individual values regarding subscriptions to TV and movie packages, it is worth determining the number of users who have both subscriptions, as this could affect potential customer churn.

In [ ]:
dual_subscription = ((df['is_tv_subscriber'] == 1) & (df['is_movie_package_subscriber'] == 1)).sum()
print('Користувачі, які мають подвійну підписку: ', dual_subscription)
# print('Users that have a dual subscription: ', dual_subscription)

In [ ]:
percentage_of_dual_subscription = round((dual_subscription / df.shape[0]) * 100,2)
print(f'{percentage_of_dual_subscription}% користувачів мають підписку як на телевізійний, так і на кінопакет.')
# print(f'{percentage_of_dual_subscription}% of users have both tv and movie-package subscriptions')

**Українська версія:**  
З огляду на те, що більше третини користувачів мають передплату як на телебачення, так і на кінопакет, що не відображено належним чином у наборі даних, для урахування цього фактора необхідно створити стовпець «подвійна передплата».

**English Version:**    
Considering the fact that more than third users have both tv and movie-package subscription that is not properly stated in the dataset, in order to keep its impact the column of dual subscription must be created.

In [ ]:
df['dual_subscriber'] = ((df['is_tv_subscriber'] == 1) & (df['is_movie_package_subscriber'] == 1)).astype('int')
df.head(-5)

**Українська версія:**   
Наступним кроком буде пошук відповідного рішення для заповнення значень NaN.

**English version:**   
Next step would be to find the proper solution for filling NaN values.

In [ ]:
df.groupby(df['remaining_contract'].isna())['churn'].mean()

In [ ]:
df.loc[df['remaining_contract'].isna(), 'is_tv_subscriber'].sum(), df.loc[df['remaining_contract'].isna(), 'is_movie_package_subscriber'].sum(), 

**Українська версія:**   
Оскільки понад 91% клієнтів із відсутнім значенням remaining_contract перейшли у відтік, цей патерн занадто сильний, щоб втрачати його через просте заповнення пропусків. Щоб зберегти цей сигнал, ми створюємо колонку has_contract_info (яка фіксує, чи були дані про контракт взагалі зафіксовані) до заповнення пропущених значень нулем. Далі створюємо колонку has_active_contract (remaining_contract > 0), щоб відрізнити клієнтів з активним контрактом від тих, чий контракт вже закінчився.  

**English version:**  
Since over 91% of customers with a missing remaining_contract value have churned, this pattern is too strong to discard through simple imputation. To preserve this signal, we introduce has_contract_info (indicating whether contract data was originally recorded) before filling missing values with 0. We then derive has_active_contract (remaining_contract > 0) to distinguish customers still under an active contract from those whose contract has expired.

In [ ]:
df['has_contract_info'] =  df['remaining_contract'].notna().astype(int)
df

In [ ]:
df['remaining_contract'] = df['remaining_contract'].fillna(0)
df['has_active_contract'] = (df['remaining_contract'] > 0).astype(int)
df.head(5)

In [ ]:
df.groupby(df['download_avg'].isna())['churn'].mean()

In [ ]:
df.groupby(df['upload_avg'].isna())['churn'].mean()

In [ ]:
df.groupby(df['download_avg'].isna())['subscription_age'].describe()

**Українська версія:**   
Рядки з відсутніми значеннями download_avg/upload_avg (381 з 72,274) мають рівень відтоку 0%, порівняно з 55.7% для клієнтів із зафіксованими даними про використання. Ці ж рядки також мають значно нижчий медіанний subscription_age (0.02 проти 1.98 років), що свідчить про те, що більшість із них — нещодавно зареєстровані клієнти, які ще не встигли згенерувати дані про використання, тому заповнення значенням 0 є семантично коректним, а не оцінкою.  

**English version:**  
Rows with missing download_avg/upload_avg values (381 out of 72,274) show a churn rate of 0%, compared to 55.7% for customers with recorded usage data. These missing rows also have a notably lower median subscription_age (0.02 vs 1.98 years), suggesting most are very recently signed-up customers who have not yet generated usage data, making 0 a semantically accurate fill value rather than an estimate.

In [ ]:
df['download_avg'] = df['download_avg'].fillna(0)
df['upload_avg'] = df['upload_avg'].fillna(0)

In [ ]:
df

In [ ]:
df.to_csv('../data/processed/df_cleaned.csv',index=False)

## **Українська версія:**   
### Візуалізація даних 

## **English version:**     
### Data Visualization 

In [ ]:
df_copy = df.copy()

### 1. Графік, що відображає частку активних користувачів та користувачів, які відмовилися від послуги компанії (The graph of The proportion of active and churned users) 

In [ ]:
churned_users = df_copy['churn'].sum()
active_users = df_copy.shape[0] - churned_users

In [ ]:
data_users = [churned_users,active_users]
labels_users = ['Відтік користувачів','Активні користувачі']
plt.title('Частка активних користувачів та користувачів, які відмовилися від послуги компанії')
plt.pie(data_users,labels=labels_users,autopct="%.2f%%",colors=['Teal','Salmon'],shadow=False, labeldistance=1.1, startangle=0, radius=1)
plt.show()

**Українська версія:**  
Кругова діаграма демонструє, що набір даних містить більше користувачів, які припинили користуватися послугами, і їхня частка становить 55,41%.
**English version:**     
The piechart illustrates that the dataset contains more churned users and their proportion contains 55.41%. 

### 2. Огляд відтоку клієнтів за типами підписки (Overview of Customer Churn by Subscription Type)

In [ ]:
rows = ['is_tv_subscriber', 'is_movie_package_subscriber', 'dual_subscriber']

x = ['Без підписки', 'З підпискою']
churn_status = ['Відсутність відтоку клієнтів', 'Відтік клієнтів']
graphichs_amount = 6
fig, axs = plt.subplots(len(rows), 2, figsize=(12,12))
for i, col in enumerate(rows):
        churn_count = df_copy.groupby(rows[i])['churn'].sum().sort_index(ascending=True)
        axs[i,0].bar(x,churn_count,label=rows[i])
        axs[i,0].set_title(f'Підписка {rows[i]}',fontsize=10)
        axs[i,0].set_xlabel(f'Кількість користувачів з {rows[i]} підпискою')
        # axs[i,0].set_xlabel(f'Users quantity with a {rows[i]} subscription') 
        axs[i,0].set_ylabel('Кількість відтоку')
        
        axs[i,1].pie(churn_count)
        axs[i,1].set_title(rows[i],fontsize=10)
        wedges, texts, autotexts = axs[i,1].pie(
            churn_count,
            autopct='%1.1f%%',  
            startangle=90
        )
        axs[i,1].legend(
            wedges, churn_status,           
            loc='upper center',
            bbox_to_anchor=(0.5, -0.05), 
            shadow=True,
            ncol=2,
            fontsize=8
        )

fig.suptitle('Огляд відтоку клієнтів за типами підписки')
plt.tight_layout()
# fig.suptitle('Overview of Customer Churn by Subscription Type ')
plt.show()

In [ ]:
df_copy['is_movie_package_subscriber'].sum(), df_copy['dual_subscriber'].sum()

**Українська версія:**  
На основі отриманих графіків можна чітко помітити, що понад 70 % користувачів із підпискою на телебачення відмовилися від телекомунікаційних послуг, тоді як 79,5 % користувачів, які мають підписку на фільми, не відмовилися від послуг, що доводить: підписка на пакет фільмів є важливою причиною збереження телекомунікаційних послуг. Крім того, абоненти, які мають підписку як на кінопакет, так і на подвійний пакет, демонструють однакові результати за відсотковою різницею, і лише 2 користувачі мають підписку на кінопакет, але не мають підписки на телебачення.

**English version:**     
Based on the graphs provided, it is clear that over 70% of users with a TV subscription canceled their telecommunications services, while 79.5% of users with a movie subscription did not cancel their services, which proves that a movie package subscription is a key factor in retaining telecommunications services. In addition, subscribers who have subscriptions to both the movie package and the dual package show identical results in terms of percentage difference, and only 2 users have a subscription to the movie package but do not have a TV subscription.

### 3. Графік взаємозв’язку між тривалістю підписки та відтоком клієнтів (The graph between subscription duration and customer churn)

In [ ]:
### Українська версія: Кількість відтоку за інтервалом тривалості підписки
# English version: Churn volume by subscription age interval

min_subscription_age = df_copy['subscription_age'].min()
max_subscription_age = df_copy['subscription_age'].max()
column_comparison = 'subscription_age'
sub_age_interval = 0.5
sub_procedural_slip = 0.2

necessary_dict = get_churn_distribution(df_copy, min_subscription_age, max_subscription_age,column_comparison,sub_age_interval,sub_procedural_slip)
list_subscription_ages,list_churn_quantity = return_results_as_lists(min_subscription_age, max_subscription_age,column_comparison, necessary_dict,sub_age_interval,sub_procedural_slip)

In [ ]:
sub_age_title = 'Кількість відтоку за інтервалом тривалості підписки'
x_sub_age = 'Тривалість підписки'

final_sub_age_graph = show_graph_and_table_churns(list_churn_quantity,list_subscription_ages,sub_age_title,x_sub_age ) 
final_sub_age_graph

**Українська версія:**      
На основі графіка видно, що найбільший рівень відтоку спостерігається в період від 0,5 до 2,5 років, після чого, починаючи з 3,0 років, відбувається його значне зниження. Отже, це свідчить про те, що позначка 3,0 роки є межею, після якої показники відтоку суттєво зменшуються.

**English version:**   
The graph shows that the highest rate of churns occurs between 0.5 and 2.5 years, followed by a significant decline starting at the 3.0-year mark. This indicates that the 3.5-year point serves as a threshold beyond which churns rates decrease substantially.

 ### 4. Графік взаємозв’язку між середнью сумою та відтоком клієнтів (The graph between the average bill amount and customer churn)

In [ ]:
df_copy['bill_avg'].max()

In [ ]:
min_bill_avg = df_copy['bill_avg'].min()
##  Українська версія: Фактичне максимальне значення bill_avg становить 406, але використовується 220, 
##                     оскільки після цієї позначки випадків відтоку клієнтів немає.
##  English version: The actual maximum value of bill_avg is 406, but 220 is used because 
##                   there are no instances of customer churn beyond this point.
max_bill_avg = 220.0
column_bill_avg = 'bill_avg'
bill_avg_interval = 10
bill_procedural_slip = 1
necessary_bill_dict = get_churn_distribution(df_copy, min_bill_avg, max_bill_avg,column_bill_avg,bill_avg_interval,bill_procedural_slip)
bill_avg_list,churn_quantity_list = return_results_as_lists(min_bill_avg, max_bill_avg,column_bill_avg, necessary_bill_dict,bill_avg_interval,bill_procedural_slip)

In [ ]:
bill_avg_title = 'Кількість відтоку за інтервалом середньої суми'
x_bill_avg = 'Середній чек'

final_bill_avg_graph = show_graph_and_table_churns(churn_quantity_list,bill_avg_list,bill_avg_title,x_bill_avg) 
final_bill_avg_graph

In [ ]:
df_copy['bill_avg'].mean()

**Українська версія:**   
Наведений нижче графік демонструє, що найбільший відсоток відтоку клієнтів (понад 91%) спостерігається серед облікових записів середнього розміру із залишком на рахунку 30 або менше, причому пік відтоку припадає на діапазон середнього розміру рахунку від 10 до 30. Отже, варто зазначити: що вищою є сума рахунку (починаючи з 30), то менша ймовірність того, що користувач скасує підписку. Крім того, середнє значення показника `bill_avg` становить 18,9 — це величина в межах діапазону від 10 до 20, на який припадає 35,58% випадків відтоку.

**English version:**     
The graph below shows that the highest customer churn rate (over 91%) occurs among mid-sized bill with a balance of 30 or less, with the peak churn observed in the 10–30 bill size range. Thus, it is worth noting that the higher the bile amount (starting from 30), the lower the likelihood of a user cancelling their subscription. Furthermore, the average value of the `bill_avg` metric is 18.9 — a figure falling within the 10–20 range, which accounts for 35.58% of churn cases.

### 5.  Графіки взаємозв’язку між тривалістю підписки та середньою сумою рахунку, з розбивкою за рівнем відтоку клієнтів (A graph showing the relationship between the duration of a subscription and the average bill amount, broken down by customer churn rate)

In [ ]:
graph = sns.relplot(x='subscription_age',y='bill_avg', hue='churn', col='churn',data=df_copy)
graph.fig.suptitle('Графік взаємозв’язку між тривалістю підписки,середньої суми рахунку та відтоком клієнтів',y=1.08)
plt.show()

**Українська версія:**   
На даному графіку не спостерігається суттєвих відмінностей (через надмірне накладання графіків) у взаємодії змінних bill_avg та subscription_age: показники відтоку клієнтів майже однакові, за винятком того, що користувачі з найдорожчими рахунками не відмовляються від послуг і продовжують користуватися телекомунікаційними послугами.

 **English version:**      
This graph shows no significant differences (because of overplotting) in the interaction between the variables `bill_avg` and `subscription_age`: customer churn rates are nearly identical, except that users with the highest bills do not cancel their services and continue to use telecommunications services.

In [ ]:
df_filtered = df_copy[df_copy['bill_avg'] <= 150]
sns.lineplot(x='bill_avg',y='subscription_age',data=df_filtered)
plt.title('Співвідношення між терміном підписки та середньою сумою рахунку')
plt.tight_layout()
plt.show()

In [ ]:
df_copy['bill_avg'].mean(), df_copy['subscription_age'].mean()

**Українська версія:**   
Графік показує середнє значення subscription_age для кожного значення bill_avg (обмежено діапазоном 0–150). Затінена область навколо лінії — це 95% довірчий інтервал, обчислений методом бутстрепінгу. У діапазоні bill_avg від 0 до ~150, де зосереджена переважна більшість клієнтів, середня тривалість підписки коливається приблизно між 1.5 та 5.5 роками, без чіткого монотонного зв'язку між сумою рахунку та тривалістю підписки — тобто клієнти з різними тарифами затримуються в компанії приблизно однаково довго. Після позначки bill_avg ≈ 150 лінія стає різкою та нестабільною, а довірчий інтервал — значно ширшим. Це не відображає реальну закономірність, а є артефактом малої вибірки: лише 93 з 72,274 клієнтів (0.13%) мають bill_avg вище 150, тому середнє значення стає надзвичайно чутливим до окремих викидів.

 **English version:**      
This chart shows the mean subscription_age for each bill_avg value, limited to the range 0–150. The shaded band around the line is a 95% confidence interval, computed through bootstrapping. The range was capped at bill_avg ≤ 150 since this covers 99.87% of all customers (72,181 out of 72,274); values above 150 represent only 93 customers and produced sharp, unreliable fluctuations due to the small sample size. Within this range, average subscription age fluctuates between roughly 1.5 and 5.5 years, with no clear monotonic relationship between bill amount and tenure — customers across different billing levels stay with the company for a similar length of time, regardless of how much they're billed.

### 6. Графіки залежності відтоку клієнтів від кількості збоїв у наданні послуг (Graph showing the relationship between customer churn and the number of service outages)

In [ ]:
### service_failure = sv
'''Українська версія:
    Будує чотири графіки для заданої ознаки (`request_column`):

    1. Використання ознаки серед УСІХ клієнтів. Показує, який відсоток усіх клієнтів мав ненульове значення
       заданої ознаки, а який — не мав.

    2. Використання ознаки серед КЛІЄНТІВ, ЯКІ ПІШЛИ. Показує, який відсоток клієнтів, що пішли, мав ненульове значення 
       заданої ознаки, а який — не мав. Тобто графік показує структуру групи клієнтів, які пішли. 
       Він НЕ показує, що ця ознака була причиною відтоку.

    3. Рівень відтоку серед клієнтів, ЯКІ МАЛИ ознаку. Показує, який відсоток клієнтів із ненульовим значенням
       заданої ознаки пішов, а який залишився активним.

    4. Рівень відтоку серед клієнтів, ЯКІ НЕ МАЛИ ознаки. Показує, який відсоток клієнтів із нульовим значенням 
       заданої ознаки пішов, а який залишився активним.
'''
feature_column_sv = 'service_failure_count'

feature_presence_labels_sv = ['Мали сервісні збої','Не мали сервісних збоїв']

customer_status_labels_sv = ['Пішли','Залишилися активними']

title_all_customers_sv = 'Сервісні збої серед усіх клієнтів'
title_churned_customers_sv = 'Сервісні збої серед клієнтів, які пішли'

title_churn_rate_with_feature_sv = 'Пішли чи залишилися: клієнти із сервісними збоями'
title_churn_rate_without_feature_sv = 'Пішли чи залишилися: клієнти без сервісних збоїв'


result_sv = show_all_churn_piecharts(df_copy,feature_column_sv,feature_presence_labels_sv,
                                     customer_status_labels_sv ,title_all_customers_sv,title_churned_customers_sv,
                                     title_churn_rate_with_feature_sv,title_churn_rate_without_feature_sv)
result_sv

**Українська версія:**     
Перші два графіки показують, що поширеність збоїв у наданні послуг є майже однаковою як серед усіх клієнтів, так і серед тих, хто відмовився від послуг: 16,42 % усіх клієнтів та 16,69 % клієнтів, які відмовилися від послуг, стикалися принаймні з одним збоєм у наданні послуг. Третій і четвертий графіки показують, що показники відтоку клієнтів також дуже схожі між цими двома групами: 56,34% серед клієнтів, які стикалися з перебоями в наданні послуг, проти 55,23% серед клієнтів, які цього не зазнали. Отже, виходячи з цих результатів, істотної різниці в рівні відтоку клієнтів між тими, хто стикався з перебоями в наданні послуг, та тими, хто не стикався, немає. Хоча більше половини клієнтів, які стикалися з перебоями в наданні послуг, зрештою відтоку (56,34 %), дуже схожа частка клієнтів, які не стикалися з такими перебоями, також відтоку (55,23 %). Отже, самі по собі ці результати не дають переконливих доказів того, що перебої в наданні послуг пов’язані з вищим ризиком відтоку клієнтів.

**English version:**      
The first two charts show that the prevalence of service failures is almost identical among all customers and among churned customers: 16.42% of all customers and 16.69% of churned customers experienced at least one service failure. The third and fourth charts show that the churn rates are also very similar between the two groups: 56.34% among customers who experienced service failures versus 55.23% among customers who did not. Therefore, based on these results, there is no substantial difference in churn rate between customers with and without service failures. Although more than half of customers who experienced service failures eventually churned (56.34%), a very similar proportion of customers without service failures also churned (55.23%). Consequently, these results alone do not provide strong evidence that service failures are associated with a higher churn risk.

In [ ]:

min_service_failure = float(df_copy['service_failure_count'].min())
max_service_failure = float(df_copy['service_failure_count'].max())
column_service_failure = 'service_failure_count'
service_failure_interval = 1
service_failure_procedural_slip = 1

necessary_service_dict = get_churn_distribution(df_copy, min_service_failure, max_service_failure,column_service_failure,service_failure_interval,service_failure_procedural_slip)
service_failure_list,churn_quantity_list_f = return_results_as_lists(min_service_failure, max_service_failure,column_service_failure, necessary_service_dict,service_failure_interval,service_failure_procedural_slip)

In [ ]:
service_failure_title = 'Кількість відтоку від кількості збоїв'
x_service_failure = 'Кількість збоїв'

table_height_service_failure = 1
final_service_failure_graph = show_graph_and_table_churns(churn_quantity_list_f,service_failure_list,service_failure_title,x_service_failure,table_height_service_failure) 
final_service_failure_graph

**Українська версія:**     
Графік демонструє, що, хоча 83,31% користувачів, які відмовилися від послуг, не стикалися з жодними збоями в їх наданні, 10,39% випадків відтоку клієнтів могли бути спричинені першим збоєм, а другий збій міг призвести до втрати 3,63% клієнтів; водночас подальші збої не мають суттєвого впливу на цей показник. Отже, це свідчить про те, що кількість збоїв у наданні послуг не є вагомою причиною для припинення користування телекомунікаційними послугами.

**English version:**      
The graph shows that while 83.31% of churned users did not have any service failures,  10.39% of customer churn cases could be caused by the first service failure, and a second failure could lead to 3.63% churns, whereas subsequent failures do not have a significant impact on this figure. Therefore, it proves that the quanity of service failures does not serve as a convenient reason to terminate telecommunication services. 

### 7. Графіки відтоку клієнтів, які мають чинні контракти (The graph of customer churn for active contracts)

In [ ]:
### active_contract = ac
feature_column_ac = 'has_active_contract'

feature_presence_labels_ac = ['Мають чинний контракт','Не мають чинний контракт']

customer_status_ac = ['Пішли','Залишилися активними']


title_all_customers_ac = 'Чинні контракти серед усіх клієнтів'
title_churned_customers_ac = 'Чинні контракти серед клієнтів, які пішли'

title_churn_rate_with_feature_ac = 'Пішли чи залишилися: клієнти із чинними контрактами'
title_churn_rate_without_feature_ac = 'Пішли чи залишилися: клієнти без чинних контрактів'

result_ac = show_all_churn_piecharts(df_copy, feature_column_ac,feature_presence_labels_ac,
                                                   customer_status_ac,title_all_customers_ac,title_churned_customers_ac,
                                                   title_churn_rate_with_feature_ac,title_churn_rate_without_feature_ac)
result_ac

**Українська версія:**     
Перша кругова діаграма показує, що 52,49 % усіх клієнтів не мають чинного договору. Друга діаграма показує, що лише 10,09 % клієнтів, які відмовилися від послуг, мали чинний договір на момент відходу, що означає: 89,91 % усіх випадків відмови припадає на клієнтів без чинних договірних зобов’язань. Крім того, третій графік ілюструє, що понад 88 % (88,23 %) клієнтів з діючими договорами не відмовилися від телекомунікаційних послуг, тоді як, навпаки, четвертий графік демонструє, що 94,93 % тих, хто не мав діючого договору, відмовилися від послуг. Це вказує на те, що відсутність діючого договору тісно пов’язана з вищою ймовірністю відтоку клієнтів. Практичний висновок полягає в тому, що найвищий ризик відтоку клієнтів у компанії зосереджений у її клієнтській базі без контрактів, і заходи з утримання клієнтів — такі як стимулювання поновлення контрактів або пропозиція строкових акцій — повинні бути спрямовані насамперед на цей сегмент.

**English version:**      
The first pie chart shows that 52.49% of all customers do not have an active contract. The second chart shows that only 10.09% of churned customers had an active contract at the time they left, meaning 89.91% of all churn comes from customers with no active contractual commitment. Additionally, the third graph illustrates that over 88% (88.23%) of customers with active contracts did not churn the telecommunication services, while on the contrary, the fourth graph demonstrates that 94.93% of those who did not have an active contract churned. This indicates that lacking an active contract is strongly associated with a higher likelihood of leaving. The practical implication is that the company's highest churn risk is concentrated in its non-contract customer base, and retention efforts — such as incentivizing contract renewal or offering term-based promotions — should be targeted primarily at this segment.

### 8. Графіки порівняння та впливу середніх показників швидкості завантаження (download) і вивантаження (upload) на кількість відтоку клієнтів. (The comparing graphs that illustrate the impact of download and upload averages metrics on customer churn)

In [ ]:
#### download_avg = da
feature_column_da = 'download_avg'

feature_presence_labels_da = ['Скористалися опцією завантаження', 'Не скористалися опцією завантаження']
customer_status_da = ['Пішли','Залишилися активними']

title_all_customers_da = 'Усі клієнти: використання опції завантажень'
title_churned_customers_da = 'Клієнти, що пішли: використання опції завантажень'
title_churn_rate_with_feature_da = 'Пішли чи залишилися: використали опцію завантаження'
title_churn_rate_without_feature_da = 'Пішли чи залишилися: НЕ використали опцію завантаження'

result_da = show_all_churn_piecharts(df_copy,feature_column_da,feature_presence_labels_da,customer_status_da,title_all_customers_da,
                                     title_churned_customers_da,title_churn_rate_with_feature_da,title_churn_rate_without_feature_da)

result_da

**Українська версія:**    
Як видно з чотирипанельної інформаційної панелі, перший графік показує, що понад 84% (84,33%) усіх користувачів скористалися опцією завантаження (`download_avg`), тоді як цей показник для клієнтів, які відмовилися від послуги, становить 72,74% — приблизно на 11,6% менше. Крім того, третій графік ілюструє, що серед користувачів, які скористалися опцією завантаження, розподіл є майже рівним: 47,79% відписалися, а 52,21% залишаються активними. З іншого боку, четвертий графік демонструє, що переважна більшість (96,43%) клієнтів, які не скористалися опцією завантаження, зрештою відписалися. Це вказує на те, що відсутність активності щодо завантажень тісно пов’язана з відмовою від послуги.

**English version:**     
Based on the four-panel dashboard, the first chart shows that over 84% (84.33%) of all users used the download option (download_avg), while the same indicator for churned customers is 72.74%—about 11.6% lower. Furthermore, the third graph illustrates that among users who did use the download option, the split is nearly even, with 47.79% churning and 52.21% remaining active. On the other hand, the fourth graph demonstrates that an overwhelming 96.43% of customers who did not use the download option ultimately churned. This indicates that a lack of engagement with downloads is strongly associated with leaving the service.

In [ ]:
df_copy['download_avg'].max()

In [ ]:
min_download_avg = df_copy['download_avg'].min()

# Українська версія: Фактичне максимальне значення параметра `download_avg` становить 4415,2, 
#                    але було обрано значення 210, оскільки після такої кількості завантажень відтік клієнтів становив менш ніж 0,05%.
# English version: The actual maximum value of the `download_avg` parameter is 4,415.2, but a value of 210 was chosen because, 
#                 after that number of downloads, the customer churn rate was less than 0.05%. 
max_download_avg = 210
column_download_avg = 'download_avg'
download_avg_interval = 5
download_avg_procedural_slip = 1

necessary_download_avg_dict = get_churn_distribution(df_copy, min_download_avg, max_download_avg,column_download_avg,download_avg_interval,download_avg_procedural_slip)
download_avg_list,churn_quantity_list_avg = return_results_as_lists(min_download_avg, max_download_avg,column_download_avg, necessary_download_avg_dict,download_avg_interval,download_avg_procedural_slip)

In [ ]:
download_avg_title = 'Кількість відтоку у порівнянні із середнім показником завантажень'
x_download_avg = 'Кількість завантажень'
table_height_download_avg = 1.5

final_download_avg_graph = show_graph_and_table_churns(churn_quantity_list_avg,download_avg_list,download_avg_title,x_download_avg,table_height_download_avg) 
final_download_avg_graph

**Українська версія:**  
Як таблиця, так і графік демонструють, що зі збільшенням кількості завантажень частка відтоку стабільно знижується

**English version:**       
Both table and graph demonstrate that with the rise of downloads quatity the % of churns steadily decreases. 

In [ ]:
## upload_avg = ua

feature_column_ua = 'upload_avg'

feature_presence_labels_ua = ['Скористалися опцією вивантаженя', 'Не скористалися опцією вивантаженя']
customer_status_ua = ['Пішли','Залишилися активними']

title_all_customers_ua = 'Усі клієнти: використання опції вивантажень'
title_churned_customers_ua = 'Клієнти, що пішли: використання опції вивантажень'

title_churn_rate_with_feature_ua = 'Пішли чи залишилися: використали опцію вивантаженя'
title_churn_rate_with_feature_ua = 'Пішли чи залишилися: НЕ використали опцію вивантаженя'

result_ua = show_all_churn_piecharts(df_copy,feature_column_ua,feature_presence_labels_da,customer_status_ac,
                                             title_all_customers_ua,title_churned_customers_ua,title_churn_rate_with_feature_ua,
                                             title_churn_rate_with_feature_ua)

result_ua

**Українська версія:**    
Оскільки показник upload_avg демонструє майже ідентичні характеристики розподілу та тенденції відтоку порівняно з download_avg (із відхиленнями менш ніж 1,6%), отримані результати збігаються з даними щодо завантаження: низький рівень використання функцій передачі даних (upload) так само пов’язаний із вищою ймовірністю відмови від послуги, що підтверджує статус загального низького обсягу передачі даних як ключового індикатора відтоку.

**English version:**  
Since upload_avg exhibits nearly identical distributional characteristics and churn trends to download_avg (with percentage variations under 1.6%), the findings mirror those of downloads: a lack of engagement with upload features is similarly associated with a higher likelihood of leaving the service, confirming that overall low data usage acts as a key churn indicator.

In [ ]:
df_copy['upload_avg'].max()

In [ ]:
min_upload_avg = df_copy['upload_avg'].min()
# Українська версія: Фактичне максимальне значення параметра `upload_avg` становить 453.3, 
#                    але було обрано значення 165, оскільки після такої кількості вивантажень відтік клієнтів відсутній.
# English version: The actual maximum value of the `download_avg` parameter is 4,415.2, but a value of 210 was chosen because, 
#                 after that number of uploads, there are no churns. 
max_upload_avg = 165.0
column_upload_avg = 'upload_avg'
upload_avg_interval = 5
upload_avg_procedural_slip = 1

necessary_upload_avg_dict = get_churn_distribution(df_copy, min_upload_avg, max_upload_avg,column_upload_avg,upload_avg_interval,upload_avg_procedural_slip)
upload_avg_list,churn_quantity_list_upload = return_results_as_lists(min_upload_avg, max_upload_avg,column_upload_avg, necessary_upload_avg_dict,upload_avg_interval,upload_avg_procedural_slip)

In [ ]:
upload_avg_title = 'Кількість відтоку у порівнянні із середнім показником вивантажень'
x_upload_avg = 'Кількість вивантажень'

final_upload_avg_graph = show_graph_and_table_churns(churn_quantity_list_upload,upload_avg_list,upload_avg_title,x_upload_avg) 
final_upload_avg_graph

**Українська версія:**  
Як таблиця, так і графік демонструють схожу загальну тенденцію до зниження показника download_avg, але з однією ключовою відмінністю: пік відтоку користувачів припадає на момент, коли середня кількість завантажень становить 5,0 (досягаючи 57,34 %), а не на нульовому рівні. Крім того, на графіку спостерігається набагато більш різке падіння: щойно показник активності завантаження досягає 20,0, рівень відтоку користувачів опускається нижче 1% (0,96%) і продовжує стабільно та поступово знижуватися.

**English version:**       
Both the table and graph demonstrate a similar overarching downward trend to download_avg, with one key difference: the peak of churn occurs when the average upload count is at 5.0 (reaching 57.34%), rather than at zero. Furthermore, the graph drops off much more steeply; once upload engagement reaches 20.0, the churn rate falls below 1% (0.96%) and continues a stable, steady decline.

### 9. Графіки відтоку клієнтів на основі перевищення ліміту завантажень (Graphs customer churn based on download over-limit feature)

In [ ]:
sns.lineplot(x='download_over_limit',y='churn',data=df_copy)
plt.title('Співвідношення між характеристикою download_over_limit та між потенційним середнім показником відтоку')
plt.tight_layout()
plt.show()

**Українська версія:**  
Лінійний графік демонструє сильний позитивний кореляційний зв’язок: зі збільшенням показника `download_over_limit` середній рівень відтоку клієнтів різко зростає — приблизно з 0,53 (при значенні 0) — і стабільно підвищується на проміжних етапах, сягаючи 1.0, коли значення показника досягає 7.

**English version:**       
The line graph shows a strong positive correlation: as the download_over_limit feature increases, the mean churn rate rises sharply from approximately 0.53 at 0, climbing steadily through the intermediate steps to reach 1.0 when the feature value hits 7.

In [ ]:
### download_over_limit = dol
feature_column_dol = 'download_over_limit'

feature_presence_labels_dol = ['Перевищели ліміт завантаження', 'Не перевищели ліміт завантаження']
customer_status_dol = ['Пішли','Залишилися активними']

title_dol = 'Усі клієнти: перевищили ліміт завантаження'
title_dol_churns = 'Клієнти, що пішли: перевищили ліміт завантаження'

title_churn_rate_with_feature_dol = 'Пішли чи залишилися: перевищели ліміт завантаження'
title_churn_rate_without_feature_dol = 'Пішли чи залишилися: НЕ перевищели ліміт завантаження'

result_dol = show_all_churn_piecharts(df_copy,feature_column_dol,feature_presence_labels_dol,customer_status_dol,
                                      title_dol,title_dol_churns,title_churn_rate_with_feature_dol,
                                      title_churn_rate_without_feature_dol)

result_dol

**Українська версія:**  
Чотирипанельний датшборд показує, що лише 5.40% усіх клієнтів перевищили ліміт завантажень, і вони становлять таку ж малу частку (8.62%) серед загальної кількості клієнтів, що пішли. Проте аналіз рівня відтоку всередині кожного сегмента розкриває важливу інсайдерську інформацію: серед клієнтів, які перевищили ліміт, аж 88.46% пішли (третій графік). Натомість серед тих, хто не перевищував ліміти, 53.53% також залишили сервіс (четвертий графік). Крім того, лінійний графік демонструє сильну позитивну кореляцію: зі збільшенням показника download_over_limit від 0 до 7 середній рівень відтоку стрімко зростає приблизно з 0.53 до 1.0. Це вказує на те, що досягнення або перевищення лімітів завантаження є серйозним джерелом невдоволення та потужним прямим індикатором відтоку клієнтів.

**English version:**       
The four-panel dashboard shows that only 5.40% of all customers exceeded their download limit, and they account for a similarly small fraction (8.62%) of total churned users. However, looking at the churn rates within each segment reveals a critical insight: among customers who did exceed their limit, 88.46% churned (third chart). Conversely, among those who stayed within their limits, 53.53% also churned (fourth chart). Furthermore, the line graph demonstrates a strong positive correlation, showing that as the download_over_limit value increases from 0 to 7, the mean churn rate surges sharply from approximately 0.53 up to 1.0. This indicates that hitting or exceeding download limits is a major friction point and a strong direct predictor of customer abandonment.

### 10. Графік кореляції між ознаками (The correlation graph between features)

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, fmt=".2f")
# plt.title("Correlation between features")
plt.title('Кореляція між ознаками')
plt.show()

In [ ]:
df = df.drop(columns=['dual_subscriber','id'])

In [ ]:
df

In [ ]:
df.to_csv('../data/processed/df_cleaned.csv',index=False)

In [ ]:
df_copy.to_csv('../data/eda_data/df_eda.csv',index=False)